In [ ]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [27]:
import sys
import re
import fitz  # PyMuPDF
import unicodedata
import json
from datetime import datetime


In [ ]:

DEBUG = True
HEADERS = ["WARM UP", "EXTRA TRAINING", "SKILL", "WOD"]

def extract_text_from_pdf(pdf_bytes, use_blocks=False):
    text = ""
    with fitz.open(stream=pdf_bytes, filetype="pdf") as doc:
        if use_blocks:
            pages_text = []
            for page in doc:
                blocks = page.get_text("blocks")
                blocks_sorted = sorted(blocks, key=lambda b: (b[1], b[0]))
                page_text = "\n".join(b[4].strip() for b in blocks_sorted if b[4].strip())
                pages_text.append(page_text)
            text = "\n\n".join(pages_text)
        else:
            for page in doc:
                text += page.get_text("text") + "\n"
    return text

def normalize_text_for_regex(text: str) -> str:
    if not text:
        return text
    text = unicodedata.normalize("NFKC", text)
    trans = {
        '\u2013': '-', '\u2014': '-', '\u2012': '-', '\u2010': '-',
        '\u2019': "'", '\u2018': "'", '\u2032': "'", '\u201A': ',', '\u2026': '...'
    }
    for k, v in trans.items():
        text = text.replace(k, v)
    text = re.sub(r'\r\n?', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{2,}', '\n\n', text)
    return text.strip()

def parse_data_e_dia(text):
    if not text:
        return None, None
    txt = unicodedata.normalize("NFKD", text).upper()
    txt_ascii = re.sub(r'[^\x00-\x7F]', '', txt)
    m = re.search(r'(\b\d{1,2}\s+[A-ZÇÀ-Ý]+)\s*\|\s*([A-ZÇÀ-Ý\s-]*FEIRA)\b', txt_ascii, re.IGNORECASE)
    if m:
        return m.group(1).strip(), m.group(2).strip()
    m2 = re.search(r'\b(S[ÈE]GUNDA|TERCA|TERÇA|QUARTA|QUINTA|SEXTA|SABADO|DOMINGO)\b', txt_ascii, re.IGNORECASE)
    if m2:
        return None, m2.group(0).strip().upper()
    return None, None

def parse_materiais(text):
    if not text:
        return []
    match = re.search(r'\bMATERIAL(?:S)?\s*:\s*([^\n]+)', text, re.IGNORECASE)
    if not match:
        return []
    materiais_str = match.group(1).strip()
    parts = []
    for chunk in re.split(r',', materiais_str):
        for sub in re.split(r'\s+e\s+', chunk, flags=re.IGNORECASE):
            item = sub.strip()
            if item:
                parts.append(item)
    seen = set(); out = []
    for it in parts:
        key = it.lower()
        if key not in seen:
            seen.add(key); out.append(it)
    return out


def extract_observacoes_from_block(block_text):
    if not block_text:
        return None


    obs_marker = re.search(r'\bOBSERVA(?:C|Ç)AO\b\s*:', block_text, re.IGNORECASE)
    if not obs_marker:
        return None


    tail = block_text[obs_marker.end():]

    first_line_match = re.match(r'\s*(.*?)(?:\n|$)', tail)
    if first_line_match:
        first_line = first_line_match.group(1).strip()
        if first_line and not re.fullmatch(r'[-\s·•\*]+', first_line):
            rest = tail[first_line_match.end():]
            lines = [ln.rstrip() for ln in rest.splitlines()]
            collected = [first_line]
            for ln in lines:
                if not ln.strip():
                    break
                if re.match(r'^\s*(?:MATERIAL(?:S)?\s*:|\b(?:' + "|".join([re.escape(h) for h in HEADERS]) + r')\b)', ln.strip(), re.IGNORECASE):
                    break
                collected.append(ln.strip())
            return "\n".join([c for c in collected if c]).strip() or None

    lines = [ln.rstrip() for ln in tail.splitlines()]
    collected = []
    max_lines = 40
    i = 0

    while i < len(lines) and i < max_lines and (not lines[i].strip() or re.fullmatch(r'[-\s·•\*]+', lines[i].strip())):
        i += 1

    while i < len(lines) and len(collected) < max_lines:
        ln = lines[i].strip()
        if not ln:
            break
        if re.match(r'^\s*(?:MATERIAL(?:S)?\s*:|\b(?:' + "|".join([re.escape(h) for h in HEADERS]) + r')\b)', ln, re.IGNORECASE):
            break
        if re.fullmatch(r'[-\s·•\*]+', ln):
            i += 1
            continue
        collected.append(ln)
        i += 1

    if collected:
        return "\n".join(collected).strip() or None

    return None

def map_observacoes_globally(full_text, partes_dict):
    if not full_text:
        return
    headers_regex = "|".join([re.escape(h) for h in HEADERS])
    header_spans = []
    for m in re.finditer(rf'(?i)({headers_regex})', full_text):
        header_spans.append((m.start(), m.end(), m.group(1).upper()))
    header_spans_sorted = sorted(header_spans, key=lambda x: x[0])
    if not header_spans_sorted:
        return

    for m in re.finditer(r'\bOBSERVA(?:C|Ç)AO\b\s*:', full_text, re.IGNORECASE):
        obs_start = m.start()
        next_header_pos = None
        for hs, he, hn in header_spans_sorted:
            if hs > obs_start:
                next_header_pos = hs
                break
        obs_end = next_header_pos if next_header_pos else min(len(full_text), obs_start + 3000)
        tail = full_text[m.end():obs_end]

        obs_text = extract_observacoes_from_block("OBSERVAÇÃO:" + tail)
        if not obs_text:
            lines = [ln.rstrip() for ln in tail.splitlines()]
            collected = []
            i = 0
            while i < len(lines) and (not lines[i].strip() or re.fullmatch(r'[-\s·•\*]+', lines[i].strip())):
                i += 1
            while i < len(lines):
                ln = lines[i].strip()
                if not ln:
                    break
                if re.match(r'^\s*(?:MATERIAL(?:S)?\s*:|\b(?:' + "|".join([re.escape(h) for h in HEADERS]) + r')\b)', ln, re.IGNORECASE):
                    break
                if re.fullmatch(r'[-\s·•\*]+', ln):
                    i += 1
                    continue
                collected.append(ln)
                i += 1
            obs_text = "\n".join(collected).strip() if collected else ""

        candidate = None
        for hs, he, hn in reversed(header_spans_sorted):
            if hs < obs_start:
                candidate = hn
                break
        if candidate is None:
            candidate = header_spans_sorted[0][2]

        if obs_text:
            cur = partes_dict.get(candidate, {}).get("observacoes")
            if not cur:
                partes_dict[candidate]["observacoes"] = obs_text
            else:
                if obs_text not in cur:
                    partes_dict[candidate]["observacoes"] = (cur + "\n\n" + obs_text).strip()

def detect_tipo_duracao_from_line(line):
    dur = None; t = None
    dur_match = re.search(r"\((\d{1,3})\s*['’]?\)|\b(\d{1,3})\s*(?:min|m)\b", line, re.IGNORECASE)
    if dur_match:
        dur = int(dur_match.group(1) or dur_match.group(2))
    tipo_match = re.search(r"(\d+\s+ROUNDS?\s+FOR\s+TIME|\d+\s+ROUNDS?|\bAMRAP\b|\bFOR TIME\b|\bEMOM\b|\bROUNDS\b)", line, re.IGNORECASE)
    if tipo_match:
        t = tipo_match.group(0).strip().upper()
    return t, dur

def parse_partes_do_treino(text, full_text_for_fallback=None):
    partes = {}
    if not text:
        for h in HEADERS:
            partes[h] = {"nomeWod": None, "tipo": h, "duracaoMinutos": None, "exercicios": [], "observacoes": None}
        return partes

    headers_pattern = "|".join([re.escape(h) for h in HEADERS])
    for header in HEADERS:
        pattern = rf"(?i){re.escape(header)}(.*?)(?=(?:{headers_pattern})|$)"
        m = re.search(pattern, text, re.DOTALL)
        if not m:
            partes[header.upper()] = {"nomeWod": None, "tipo": header.upper(), "duracaoMinutos": None, "exercicios": [], "observacoes": None}
            continue

        bloco = m.group(1).strip()
        if DEBUG:
            print("\n" + "-"*30)
            print(f"DEBUG: bloco para header [{header}]:\n{repr(bloco[:1000])}\n")

        lines = [ln.strip() for ln in bloco.splitlines() if ln.strip()]
        header_line = lines[0] if lines else ""

        nome = None; tipo = None; duracao = None

        tipo, duracao = detect_tipo_duracao_from_line(header_line)
        header_consumed_second_line = False
        if (not tipo or not duracao) and len(lines) > 1:
            t2, d2 = detect_tipo_duracao_from_line(lines[1])
            if t2 or d2:
                if not tipo and t2: tipo = t2
                if not duracao and d2: duracao = d2
                header_consumed_second_line = True

        if '-' in header_line:
            left, right = [p.strip() for p in header_line.split('-', 1)]
            if any(k in right.upper() for k in ("AMRAP", "FOR TIME", "EMOM", "ROUNDS")):
                nome = left or None
            else:
                nome = right or (left or None)
        else:
            if tipo and header_consumed_second_line and len(lines) > 1:
                nome = header_line
            else:
                if not tipo:
                    nome = header_line
                else:
                    if len(lines) > 1 and not header_consumed_second_line:
                        possible_name = lines[1]
                        ttmp, dtmp = detect_tipo_duracao_from_line(possible_name)
                        if not (ttmp or dtmp):
                            nome = possible_name
                        else:
                            nome = header_line

        if nome:
            nome = nome.strip()
            if nome == "": nome = None

        start_idx = 2 if header_consumed_second_line and len(lines) >= 2 else (1 if len(lines) > 1 else 0)
        exercicios = []
        for ln in lines[start_idx:]:
            if re.match(r'\bOBSERVA(?:C|Ç)AO\b\s*:', ln, re.IGNORECASE) or re.match(r'\bMATERIAL(?:S)?\b\s*:', ln, re.IGNORECASE):
                continue
            if re.match(r'^\s*(?:\d+\s+ROUNDS?\b|AMRAP\b|FOR TIME\b|EMOM\b|ROUNDS\b)(.*\(\d+\))?', ln, re.IGNORECASE):
                continue
            if re.match(r'^\d{1,3}\b.*', ln) or re.match(r'^\d{1,3}\s*m\b', ln, re.IGNORECASE) or re.search(r'[A-Za-z]+\s*\(.*\d', ln):
                exercicios.append(ln); continue
            mnum = re.match(r'^(\d+)\s+([A-Za-z].+)$', ln)
            if mnum:
                exercicios.append(ln); continue
            if re.search(r'\b(KG|Kg|kg|REPS|reps|REP|m\b|metros|metros)\b', ln, re.IGNORECASE):
                exercicios.append(ln); continue

        observacoes = extract_observacoes_from_block(bloco)
        partes[header.upper()] = {"nomeWod": nome or None, "tipo": tipo or header.upper(), "duracaoMinutos": duracao, "exercicios": exercicios, "observacoes": observacoes}

    if any(partes[h]["observacoes"] is None for h in partes) and full_text_for_fallback:
        map_observacoes_globally(full_text_for_fallback, partes)

    return partes

def run_local_parser(file_path, use_blocks=False):
    TEST_BOX_ID = "box_olympus_crossfit"
    TEST_DATA_STR = "2025-11-03"

    print(f"--- Iniciando parsing local do arquivo: {file_path} ---")
    try:
        with open(file_path, 'rb') as f:
            pdf_bytes = f.read()
    except FileNotFoundError:
        print(f"Erro: Arquivo '{file_path}' não encontrado."); return
    except Exception as e:
        print(f"Erro ao ler o arquivo: {e}"); return

    raw_text = extract_text_from_pdf(pdf_bytes, use_blocks=use_blocks)
    raw_text_normalized = normalize_text_for_regex(raw_text)

    try:
        data_datetime = datetime.strptime(TEST_DATA_STR, "%Y-%m-%d")
        data_para_json = data_datetime.isoformat() + "Z"
    except Exception:
        data_para_json = TEST_DATA_STR

    data_pdf, dia_semana_pdf = parse_data_e_dia(raw_text_normalized)

    json_output = {
        "boxId": TEST_BOX_ID,
        "data": data_para_json,
        "diaDaSemana": dia_semana_pdf or "Dia não encontrado",
        "materiais": parse_materiais(raw_text_normalized),
        "partesDoTreino": parse_partes_do_treino(raw_text_normalized, full_text_for_fallback=raw_text_normalized)
    }

    print("--- Resultado do Parsing (JSON) ---")
    print(json.dumps(json_output, indent=2, ensure_ascii=False))
    return json_output


In [29]:

pdf_para_testar = "Cronograma 11 - Novembro 2025_removed.pdf"
resultado = run_local_parser(pdf_para_testar, use_blocks=False)


--- Iniciando parsing local do arquivo: Cronograma 11 - Novembro 2025_removed.pdf ---

------------------------------
DEBUG: bloco para header [WARM UP]:
"- AMRAP (5') \n05 Muscle clean \n05 Shoulder press \n10 Sprawl \n10 Air squat \n10 Sit up \n \nOBSERVAÇÃO: \n-"


------------------------------
DEBUG: bloco para header [EXTRA TRAINING]:
"- 3 ROUNDS FOR TIME (13') \n300m run \n50m lunge (50 Lunges) \n \nOBSERVAÇÃO: \n-"


------------------------------
DEBUG: bloco para header [SKILL]:
'- POWER CLEAN'


------------------------------
DEBUG: bloco para header [WOD]:
"- IN NAME OF GODS \nAMRAP (12') \n \n09 Power clean (80Kg|55Kg) \n08 Shoulder to overhead \n05 Burpee \n \nMATERIAL: Munhequeira \n \nOBSERVAÇÃO:"

--- Resultado do Parsing (JSON) ---
{
  "boxId": "box_olympus_crossfit",
  "data": "2025-11-03T00:00:00Z",
  "diaDaSemana": "SEGUNDA FEIRA",
  "materiais": [
    "Munhequeira"
  ],
  "partesDoTreino": {
    "WARM UP": {
      "nomeWod": null,
      "tipo": "AMRAP",
      "dur